# Depth Comparison Experiments

This notebook implements derived comparison analysis at depth 1 and depth 2,
based on the pairwise comparison infrastructure in `comfyui_image_scorer`.

- **Depth 1**: For each element, compares against all better and worse elements,
  accumulating `1/4` per pair. After all elements, pairs with total > 1 indicate
  at least 4 independent elements confirming the relationship.
- **Depth 2**: For each actual comparison in the database, compares all
  elements better than the winner and worse than the loser, accumulating
  `1/16` per pair. Pairs with total > 1 indicate at least 16 depth-2
  confirmations.

Formula: `comparison_counts[pair] += 1 / (base ** depth)` where `base = 4`

All tqdm progress bars use `delay=3` matching the codebase style.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone

from tqdm import tqdm as tqdm_desc

from comfyui_image_scorer.domain.graph.chain_manager import (
    ChainManager,
    parse_comparison,
)
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    SQLiteComparisonsRepository,
)


from typing import Any
from comfyui_image_scorer.infrastructure.persistence.images_repository import (
    update_image_rating_state,
    get_image,
)
from comfyui_image_scorer.domain.comparison.comparison_recorder import (
    update_scores_after_comparison,
)


In [ ]:
# Initialize ChainManager from existing comparisons and DB repository (DB-only writes, no file touches)
_BASE = 2
_THRESHOLD = 1.0  # confirmation cutoff for adding a pair to the DB
comparison_repo = SQLiteComparisonsRepository()
all_comparisons = comparison_repo.get_all_comparisons()

chain_mgr = ChainManager()
chain_mgr.build(comparisons=all_comparisons)

all_filenames = list(chain_mgr.get_all_filenames())
print(f"Loaded {len(all_filenames)} images from comparison graph")

print(f"Top nodes: {len(chain_mgr.get_top_nodes())}")
print(f"Bottom nodes: {len(chain_mgr.get_bottom_nodes())}")
print(f"Chains built: {chain_mgr.get_min_chain_count()}")


In [ ]:
# Depth 1: For each element, accumulate 1/4 per (better, worse) pair
comparison_counts: dict[tuple[str, str], float] = defaultdict(float)

print("=== Depth 1 ===")
weight = 1 / (_BASE)  # 1/base^depth
for elem in tqdm_desc(
    all_filenames, desc="Depth 1 elements", total=len(all_filenames), delay=3
):
    better_list = chain_mgr.get_better_than(elem)
    worse_list = chain_mgr.get_worse_than(elem)

    pair_list: list[tuple[str, str]] = [
        (better, worse) for better in better_list for worse in worse_list
    ]

    for pair in pair_list:
        comparison_counts[pair] += weight


In [ ]:
# Depth 2: For each actual comparison in the DB, accumulate 1/16
print("\n=== Depth 2 ===")

weight: float = 1 / (_BASE**2)

for comp in tqdm_desc(
    all_comparisons, desc="Depth 2 pairs", total=len(all_comparisons), delay=3
):
    _, _, winner, loser = parse_comparison(comp)
    better_than_winner: list[str] = chain_mgr.get_better_than(winner)
    worse_than_loser: list[str] = chain_mgr.get_worse_than(loser)

    pair_list: list[tuple[str, str]] = [
        (better, worse) for better in better_than_winner for worse in worse_than_loser
    ]

    for pair in pair_list:
        comparison_counts[pair] += weight

# since depth 2 also passes through depth 1, we need to adjust the numbers by removig the duplicates
for pair in comparison_counts:
    comparison_counts[pair] -= weight  # remove depth 2 duplicates from depth 1 counts


In [ ]:
# Filter after both depths complete (shared comparison_counts)
# near-miss: below threshold by at most one depth-1 step, measured
# in rounded depth-2 units to stay exact on float sums
confirmed_pairs = [
    (pair, count) for pair, count in comparison_counts.items() if count >= _THRESHOLD
]
near_miss_pairs = [
    (pair, count)
    for pair, count in comparison_counts.items()
    if 1 <= round((_THRESHOLD - count) * _BASE * _BASE) <= _BASE
]

total_pairs = len(comparison_counts)
confirmed_count = len(confirmed_pairs)

print("\n=== Filter Results ===")
print(f"Total pairs analyzed: {total_pairs}")
print(f"Pairs with count >= {_THRESHOLD}: {confirmed_count}")
print(
    f"Near-miss pairs ({round(_THRESHOLD - 1/_BASE, 3)} <= count < {_THRESHOLD}): "
    f"{len(near_miss_pairs)}"
)
print(f"Percentage confirmed: {confirmed_count/len(all_comparisons)*100:.1f}%")

print(f"confirmed_pairs:{confirmed_pairs[:10]} ...")  # Show first 10 for brevity


In [ ]:
# only add the nodes with the highest sigma, over the threshold, that are not already in database
sigma_threshold = 2.5
comparison_threshold = 10
removed_sigma = 0
remove_existing = 0
remove_completed = 0


for pair, count in confirmed_pairs.copy():
    if comparison_repo.comparison_exists_for_pair(*pair):
        confirmed_pairs.remove((pair, count))
        remove_existing += 1
        continue

    node_a = get_image(pair[0])
    node_b = get_image(pair[1])

    if (
        float(node_a["rating_sigma"]) < sigma_threshold
        or float(node_b["rating_sigma"]) < sigma_threshold
    ):
        confirmed_pairs.remove((pair, count))
        removed_sigma += 1
        continue

    if (
        int(node_a["comparison_count"]) > comparison_threshold
        or int(node_b["comparison_count"]) > comparison_threshold
    ):
        confirmed_pairs.remove((pair, count))
        remove_completed += 1


# print stats
print(f"Remaining confirmed pairs: {len(confirmed_pairs)}")
print(f"Removed pairs already in DB: {remove_existing}")
print(f"Removed pairs due to low sigma: {removed_sigma}")
print(f"Removed pairs with completed comparisons: {remove_completed}")


In [ ]:
# Pair-level summary stats
count_dist = Counter(round(c, 3) for _, c in confirmed_pairs)
print("Confirmed-pair count distribution:", dict(sorted(count_dist.items())))

already_in_db = sum(
    1
    for (a, b), _ in confirmed_pairs
    if comparison_repo.comparison_exists_for_pair(a, b)
)
print(f"Confirmed pairs already in DB: {already_in_db}/{len(confirmed_pairs)}")

print(f"\nNear-miss pairs: {len(near_miss_pairs)}")
for pair, count in sorted(near_miss_pairs, key=lambda x: -x[1])[:10]:
    print(f"  {count:.3f}  {pair[0]} -> {pair[1]}")


In [ ]:
# Show 5 random pairs per list (winner left, loser right)
import random

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

_SAMPLES_PER_LIST = 10
_SIDE_STYLES = (("WINNER", "tab:green"), ("LOSER", "tab:red"))


def show_pair_samples(pairs, title):
    if not pairs:
        print(f"{title}: nothing to show")
        return
    sample = random.sample(pairs, min(_SAMPLES_PER_LIST, len(pairs)))
    fig, axes = plt.subplots(
        len(sample), 2, figsize=(9, 4 * len(sample)), squeeze=False
    )
    fig.suptitle(title)
    for row, ((winner, loser), count) in enumerate(sample):
        for col, filename in enumerate((winner, loser)):
            side_label, side_color = _SIDE_STYLES[col]
            ax = axes[row][col]
            path = find_image_path(filename)
            if path is None:
                ax.text(
                    0.5,
                    0.5,
                    f"not found:\n{filename}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )
            else:
                ax.imshow(mpimg.imread(path))
            header = f"{side_label} #{row + 1}"
            if col == 0:
                header += f" | count={count:.3f}"
            ax.set_title(f"{header}\n{filename}", fontsize=8, color=side_color)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(side_color)
                spine.set_linewidth(3)
    fig.subplots_adjust(hspace=0.45, wspace=0.08)
    plt.show()


show_pair_samples(confirmed_pairs, "Confirmed pairs")
show_pair_samples(near_miss_pairs, "Near-miss pairs")


In [ ]:
def _persist_image_state(filename: str, data: dict[str, Any]) -> bool:
    return update_image_rating_state(
        filename=filename,
        score=float(data["score"]),
        rating_mu=float(data["rating_mu"]),
        rating_sigma=float(data["rating_sigma"]),
        comparison_count=int(data["comparison_count"]),
        touch_timestamp=True,
    )


In [ ]:
# DB-Only add confirmed pairs (NO file touches) -- with dry run option
print("\n=== Adding to database (DB-only, no files) ===")


dry_run = False  # Set to False to actually write to DB
if not dry_run:
    added_count = 0
    for pair, count in tqdm_desc(
        confirmed_pairs, desc="Adding to DB", total=len(confirmed_pairs), delay=3
    ):
        comparison_id = comparison_repo.add_comparison(
            filename_a=pair[0],
            filename_b=pair[1],
            winner=pair[0],
            weight=1.0,
            transitive_depth=0,
            timestamp=datetime.now(timezone.utc).isoformat(),
        )
        winner_data = get_image(pair[0])
        loser_data = get_image(pair[1])
        winner_data, loser_data = update_scores_after_comparison(
            winner_data, loser_data
        )
        _persist_image_state(pair[0], winner_data)
        _persist_image_state(pair[1], loser_data)

        added_count += 1

    print(f"\nAdded {added_count} confirmed pairs to database")
print("Experiment complete - no files were modified")
